# NeuralEvolve on Kaggle

Runs the **GPU training** steps of NeuralEvolve on Kaggle's free GPU (about 30 GPU hours per week).
It clones the repository and runs the same scripts as on the Mac; nothing is copied into this notebook.

**How to run (recommended: in the background)**

1. *Settings* (right panel): **Accelerator → GPU T4 x2** (or P100) and **Internet → on**.
2. *Add Input* → your private dataset with the files from `runs/` on the Mac (see the table below).
3. Pick the `TASK` in the next cell.
4. **Save Version → Save & Run All (Commit)**. Kaggle runs everything on its servers, even with the browser closed
   (up to 12 h). When it finishes, open the version → *Output* → download `results.zip`.

| `TASK` | What it trains | Input files needed | Time on a T4 |
|---|---|---|---|
| `"rq5"` | RQ5: 15 networks from scratch (`RQ5_EPOCHS`) | `accuracy_dataset_main.csv` | ~1.3 h at 15 epochs, ~3.5 h at 40 |
| `"final"` | the selected final-front networks from scratch | `front_selected.csv` | ~2.5 h for 12 |
| `"baseline"` | MobileNetV2 baselines from scratch | none | ~1-2 h |
| `"supernet"` | supernet (50 epochs) + accuracy dataset + accuracy predictor | none | ~2.5 h |

Already finished results in the input (e.g. a partial `standalone.csv`) are picked up, so a run continues
where an earlier one stopped.

Core ML latency, the search experiments and the iPhone steps stay on the Mac.

## Settings

In [ ]:
TASK = "rq5"          # "rq5", "final", "baseline" or "supernet"
RQ5_EPOCHS = 40       # from-scratch epochs per RQ5 network (15 = quick, 40 = fair to large networks)

REPO_URL = "https://github.com/VagifAlyv/NeuralEvolve.git"
BRANCH = "main"
CODE_DIR = "/tmp/NeuralEvolve"          # code + data: not saved with the version
OUT_DIR = "/kaggle/working"             # saved with the version: only results go here
WORKERS = 4                             # Kaggle has 4 CPU cores

## Setup

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2 (needs a phone-verified account)"
print(torch.cuda.get_device_name(0), "| PyTorch", torch.__version__)

In [ ]:
import os, shutil, subprocess
if not os.path.isdir(f"{CODE_DIR}/.git"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, CODE_DIR], check=True)
print(subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!pip install -q "pymoo>=0.6.1"

In [ ]:
# runs/ lives in the saved output folder, so results are kept with the version
RUNS = f"{OUT_DIR}/runs"
os.makedirs(RUNS, exist_ok=True)
if not os.path.islink(f"{CODE_DIR}/runs"):
    os.symlink(RUNS, f"{CODE_DIR}/runs")

# Copy input files (from an attached Kaggle dataset) to where the scripts expect them
WANTED = {
    "accuracy_dataset_main.csv": "accuracy_dataset_main.csv",
    "front_selected.csv": "final/front_selected.csv",     # ~12 networks picked from the final front
    "standalone.csv": "rq5/standalone.csv",              # partial RQ5 results -> resume
    "standalone_40ep.csv": "rq5/standalone_40ep.csv",
    "front_trained.csv": "final/front_trained.csv",      # partial final results -> resume
    "baseline_trained.csv": "final/baseline_trained.csv",
    "main.pt": "supernet/main.pt",                        # partial supernet -> resume
    "main_log.csv": "supernet/main_log.csv",
}
for root, _, files in os.walk("/kaggle/input"):
    for name in files:
        if name in WANTED:
            dst = f"{RUNS}/{WANTED[name]}"
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if not os.path.exists(dst):
                shutil.copy(os.path.join(root, name), dst)
                print("input:", WANTED[name])
%cd {CODE_DIR}
!python genome.py

## Run the task

In [ ]:
def need(path):
    assert os.path.exists(f"runs/{path}"), f"runs/{path} missing: add it to the input dataset (Add Input)"

if TASK == "rq5":
    need("accuracy_dataset_main.csv")
    rq5_out = "runs/rq5/standalone.csv" if RQ5_EPOCHS == 15 else f"runs/rq5/standalone_{RQ5_EPOCHS}ep.csv"
    !python -u train_standalone.py --sample runs/accuracy_dataset_main.csv --k 15 --epochs {RQ5_EPOCHS} --workers {WORKERS} --out {rq5_out}
elif TASK == "final":
    need("final/front_selected.csv")
    !python -u train_standalone.py --from-csv runs/final/front_selected.csv --epochs 40 --workers {WORKERS} --out runs/final/front_trained.csv
elif TASK == "baseline":
    !python -u train_standalone.py --baseline --epochs 40 --workers {WORKERS} --out runs/final/baseline_trained.csv
elif TASK == "supernet":
    !python -u supernet.py --epochs 50 --batch-size 128 --lr 0.1 --workers {WORKERS} --tag main
    !python -u evaluate.py collect --n 1000 --workers {WORKERS}
    !python predictors.py accuracy
else:
    raise ValueError(f"unknown TASK {TASK!r}")

## Pack the results

In [ ]:
# One zip to download; unzip it inside the project folder on the Mac (files land in runs/)
%cd {OUT_DIR}
!rm -f results.zip && zip -r -q results.zip runs -x "runs/cache.sqlite" && ls -lh results.zip
!find runs -type f | sort